# Verificação rápida dos dados

In [1]:
from IPython.display import display
import duckdb
import pandas as pd
import os

PARQUET_BRONZE_DIR = "../data/02-bronze"
PARQUET_SILVER_DIR = "../data/03-silver"
PARQUET_GOLD_DIR = "../data/04-gold"

OUTPUT_DIR = "../data/05-output/notebooks"

## Funções auxiliares

In [2]:
# =========================================================
# 1. Leitura
# =========================================================

def get_parquet_files(dir_path):
    """Retorna uma lista de arquivos Parquet em um diretório específico."""

    if not os.path.exists(dir_path):
        return []

    return [
        os.path.join(dir_path, x)
        for x in os.listdir(dir_path)
        if x.endswith(".parquet")
    ]

def load_parquet_dataset(file_path, sample_size=10):

    df_full = duckdb.sql(
        f"""
        SELECT *
        FROM read_parquet('{file_path}')
        """
    ).df()

    df_sample = duckdb.sql(
        f"""
        SELECT *
        FROM read_parquet('{file_path}')
        USING SAMPLE {sample_size} ROWS
        """
    ).df()

    return {
        "file_path": file_path,
        "file_name": os.path.basename(file_path),
        "df_full": df_full,
        "df_sample": df_sample
    }


# =========================================================
# 2. Profiling / Estatísticas
# =========================================================

def generate_profile(dataset):

    df = dataset["df_full"]

    profile = {
        "shape": df.shape,
        "columns": [],
        "numeric_stats": None,
        "categorical_summary": None,
        "categorical_details": None
    }

    # =====================================================
    # Colunas
    # =====================================================

    for col in df.columns:

        null_count = df[col].isnull().sum()

        profile["columns"].append({
            "column": col,
            "dtype": str(df[col].dtype),
            "null_count": int(null_count),
            "null_percent": round(
                (null_count / len(df)) * 100,
                2
            )
        })

    # =====================================================
    # Numéricas
    # =====================================================

    df_numeric = df.select_dtypes(include=["number"])

    if not df_numeric.empty:

        numeric_stats = df_numeric.describe().T

        numeric_stats["null_count"] = (
            df_numeric.isnull().sum()
        )

        numeric_stats["null_percent"] = (
            df_numeric.isnull().sum() / len(df)
        ) * 100

        profile["numeric_stats"] = numeric_stats

    # =====================================================
    # Categóricas
    # =====================================================

    df_cat = df.select_dtypes(
        include=["object", "category", "string"]
    )

    if not df_cat.empty:

        # =================================================
        # Resumo geral das colunas categóricas
        # =================================================

        categorical_summary = []

        # =================================================
        # Detalhamento dos valores
        # =================================================

        categorical_details = []

        for col in df_cat.columns:

            null_count = df_cat[col].isnull().sum()

            # =============================================
            # Resumo
            # =============================================

            categorical_summary.append({

                "column": col,

                "dtype": str(df_cat[col].dtype),

                "null_count": int(null_count),

                "null_percent": round(
                    (null_count / len(df)) * 100,
                    2
                ),

                "unique_values": int(
                    df_cat[col].nunique(dropna=False)
                )
            })

            # =============================================
            # Top 10 valores
            # =============================================

            value_counts = (
                df_cat[col]
                .value_counts(dropna=False)
                .head(10)
            )

            for value, count in value_counts.items():

                percent = (count / len(df)) * 100

                categorical_details.append({

                    "column": col,

                    "value": str(value),

                    "count": int(count),

                    "percent": round(percent, 2)
                })

        profile["categorical_summary"] = pd.DataFrame(
            categorical_summary
        )

        profile["categorical_details"] = pd.DataFrame(
            categorical_details
        )

    return profile

# =========================================================
# 3. Visualização
# =========================================================

def display_profile(dataset, profile):

    print(f"\nArquivo: {dataset['file_name']}")

    print("\nShape:")
    print(profile["shape"])

    print("\nAmostra:")
    display(dataset["df_sample"])

    if profile["numeric_stats"] is not None:

        print("\nEstatísticas Numéricas")
        display(profile["numeric_stats"])

    if profile["categorical_summary"] is not None:

        print("\nResumo categórico")
        display(profile["categorical_summary"])

    if profile["categorical_details"] is not None:

        print("\nTop valores categóricos")
        display(profile["categorical_details"])

        

# =========================================================
# 4. Persistência
# =========================================================

def save_profile_txt(dataset, profile, output_dir, camada):

    os.makedirs(output_dir, exist_ok=True)

    output_file = os.path.join(
        output_dir,
        f"{camada}_{dataset['file_name'].replace('.parquet', '.txt')}"
    )

    with open(output_file, "w", encoding="utf-8") as f:

        f.write("=" * 100 + "\n")
        f.write(f"ARQUIVO: {dataset['file_name']}\n")
        f.write("=" * 100 + "\n\n")

        # =================================================
        # Shape
        # =================================================

        f.write("SHAPE\n")
        f.write("-" * 100 + "\n")
        f.write(f"{profile['shape']}\n\n")

        # =================================================
        # Colunas
        # =================================================

        f.write("COLUNAS\n")
        f.write("-" * 100 + "\n")

        for col in profile["columns"]:

            f.write(
                f"{col['column']:<30} | "
                f"{col['dtype']:<15} | "
                f"Nulos: {col['null_count']:<10} | "
                f"% Null: {col['null_percent']:>6.2f}%\n"
            )

        # =================================================
        # Estatísticas numéricas
        # =================================================

        if profile["numeric_stats"] is not None:

            f.write("\n\nESTATÍSTICAS NUMÉRICAS\n")
            f.write("-" * 100 + "\n")

            f.write(
                profile["numeric_stats"].to_string()
            )

        # =================================================
        # Resumo categórico
        # =================================================

        if profile["categorical_summary"] is not None:

            f.write("\n\nRESUMO CATEGÓRICO\n")
            f.write("-" * 100 + "\n")

            f.write(
                profile["categorical_summary"].to_string(
                    index=False
                )
            )

        # =================================================
        # Detalhamento categórico
        # =================================================

        if profile["categorical_details"] is not None:

            f.write("\n\nTOP VALORES CATEGÓRICOS\n")
            f.write("-" * 100 + "\n")

            f.write(
                profile["categorical_details"].to_string(
                    index=False
                )
            )

        # =================================================
        # Sample
        # =================================================

        f.write("\n\nAMOSTRA\n")
        f.write("-" * 100 + "\n")

        f.write(
            dataset["df_sample"].to_string(index=False)
        )

    print(f"Arquivo salvo: {output_file}")

# =========================================================
# Pipeline de análise
# =========================================================

def process_layer(layer_name, layer_dir, output_dir):

    print(
        "\n",
        "=" * 50,
        f"\n\t Verificando arquivos {layer_name.upper()} \n",
        "=" * 50
    )

    files = get_parquet_files(layer_dir)

    if not files:
        print(f"Nenhum arquivo encontrado em: {layer_dir}")
        return

    for file in files:

        try:

            dataset = load_parquet_dataset(file)

            profile = generate_profile(dataset)

            display_profile(dataset, profile)

            save_profile_txt(
                dataset=dataset,
                profile=profile,
                output_dir=output_dir,
                camada=layer_name
            )

        except Exception as e:

            print(f"Erro ao processar {file}")
            print(f"Detalhes: {e}")

## Visualiza todos os dados

In [3]:
# =========================================================
# Execução
# =========================================================

LAYERS = {
    "02-bronze": PARQUET_BRONZE_DIR,
    "03-silver": PARQUET_SILVER_DIR,
    "04-gold": PARQUET_GOLD_DIR
}

for layer_name, layer_dir in LAYERS.items():

    process_layer(
        layer_name=layer_name,
        layer_dir=layer_dir,
        output_dir=OUTPUT_DIR
    )


	 Verificando arquivos 02-BRONZE 

Arquivo: despesa.parquet

Shape:
(196033, 30)

Amostra:


,DT_GERACAO,HH_GERACAO,AA_EXERCICIO,TP_DESPESA,CD_TP_ESFERA_PARTIDARIA,DS_TP_ESFERA_PARTIDARIA,SG_UF,CD_MUNICIPIO,NM_MUNICIPIO,NR_ZONA,...,NR_CPF_CNPJ_FORNECEDOR,NM_FORNECEDOR,DS_GASTO,DT_PAGAMENTO,VR_GASTO,VR_PAGAMENTO,VR_DOCUMENTO,CD_FONTE_DESPESA,DS_FONTE_DESPESA,SQ_DESPESA
0,25/04/2026,16:23:25,2025,D,2,Estadual,PE,-1,#NULO#,-1,...,06348660000144,Direção Municipal/Comissão Provisória - PP - G...,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - FUNDO P...,17/04/2025,20000,20000,20000,1,Fundo Partidário,-1
1,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,10235570000114,Direção Estadual/Distrital - PT - PARÁ,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - FUNDO P...,28/07/2025,"169042,22","169042,22","169042,22",1,Fundo Partidário,-1
2,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,13477302000105,Direção Estadual/Distrital - PT - BAHIA,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - OUTROS ...,29/08/2025,"18,3","18,3","18,3",2,Outros Recursos,-1
3,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,24472136000162,Direção Estadual/Distrital - PT - ALAGOAS,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - OUTROS ...,10/04/2025,"110,61","110,61","110,61",2,Outros Recursos,-1
4,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,04516811000109,Direção Estadual/Distrital - PP - ACRE,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - FUNDO P...,01/09/2025,95000,95000,95000,1,Fundo Partidário,-1
5,25/04/2026,16:23:25,2025,D,0,Nacional,BR,-1,#NULO#,-1,...,29217381000147,Direção Estadual/Distrital - NOVO - PERNAMBUCO,TRANSFERÊNCIAS FINANCEIRAS EFETUADAS - OUTROS ...,26/02/2025,174,174,174,2,Outros Recursos,-1
6,25/04/2026,16:23:25,2025,G,1,Distrital,DF,-1,#NULO#,-1,...,15111975000164,IUGU INSTITUICAO DE PAGAMENTO S.A.,DESPESAS FINANCEIRAS - TAXAS DE ADMINISTRAÇÃO ...,05/11/2025,2,2,2,2,Outros Recursos,4051990
7,25/04/2026,16:23:25,2025,G,2,Estadual,SP,-1,#NULO#,-1,...,40432544000147,CLARO S/A,TELECOMUNICAÇÕES E INTERNET - ORDINÁRIAS,02/06/2025,"429,29","429,29","429,29",1,Fundo Partidário,3909247
8,25/04/2026,16:23:25,2025,G,2,Estadual,PR,-1,#NULO#,-1,...,45771358000156,STRIDE AGENCIA LTDA,OUTRAS DESPESAS GERAIS - ORDINÁRIAS,02/07/2025,39000,39000,39000,1,Fundo Partidário,3899107
9,25/04/2026,16:23:25,2025,G,2,Estadual,PE,-1,#NULO#,-1,...,09759606000180,GRANDE RECIFE CONSORCIO,PESSOAL - AUXÍLIO-TRANSPORTE - ORDINÁRIAS,11/09/2025,"231,8","231,8","231,8",1,Fundo Partidário,3970127



Resumo categórico


,column,dtype,null_count,null_percent,unique_values
0,DT_GERACAO,str,0,0.0,1
1,HH_GERACAO,str,0,0.0,2
2,AA_EXERCICIO,str,0,0.0,1
3,TP_DESPESA,str,0,0.0,5
4,CD_TP_ESFERA_PARTIDARIA,str,0,0.0,5
5,DS_TP_ESFERA_PARTIDARIA,str,0,0.0,5
6,SG_UF,str,0,0.0,28
7,CD_MUNICIPIO,str,0,0.0,2053
8,NM_MUNICIPIO,str,0,0.0,2015
9,NR_ZONA,str,0,0.0,2



Top valores categóricos


...


Resumo categórico


,column,dtype,null_count,null_percent,unique_values
0,sg_partido,str,0,0.0,34



Top valores categóricos


,column,value,count,percent
0,sg_partido,AGIR,1,2.94
1,sg_partido,AVANTE,1,2.94
2,sg_partido,CIDADANIA,1,2.94
3,sg_partido,DC,1,2.94
4,sg_partido,DEM,1,2.94
5,sg_partido,DEMOCRATA,1,2.94
6,sg_partido,MDB,1,2.94
7,sg_partido,MOBILIZA,1,2.94
8,sg_partido,NOVO,1,2.94
9,sg_partido,PATRIOTA,1,2.94


Arquivo salvo: ../data/05-output/notebooks/04-gold_partido_ano_receita.txt
